# Lab 4 — Multi-Agent Conflict Resolution

**Goal:** When two agents disagree, a human arbitrates. Both agents then update based on the decision.

**Real-world use:** Code review agents disagreeing on style, financial agents disagreeing on risk, medical agents disagreeing on diagnosis. A human makes the call — and the agents learn from it.

**Pattern:**
```
Agent A generates opinion
Agent B generates opinion
    ↓  (they disagree)
Human sees both opinions, picks winner or provides synthesis
    ↓
Both agents receive the human decision
Final answer generated incorporating the decision
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()
print('Ready ✓')

## 1. Two agents with different personas

In [ ]:
class AgentOpinion(BaseModel):
    recommendation: str
    reasoning: str
    confidence: float


def agent_opinion(persona: str, question: str) -> AgentOpinion:
    resp = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': persona},
            {'role': 'user', 'content': question},
        ],
        response_format=AgentOpinion,
    )
    return resp.choices[0].message.parsed


QUESTION = 'Should we launch the product in Q3 or wait until Q4 for the holiday season?'

conservative = agent_opinion(
    'You are a conservative business analyst focused on minimising risk and ensuring readiness.',
    QUESTION,
)

aggressive = agent_opinion(
    'You are an aggressive growth strategist focused on capturing market share first.',
    QUESTION,
)

print('CONSERVATIVE AGENT:')
print(f'  Recommendation: {conservative.recommendation}')
print(f'  Confidence:     {conservative.confidence:.0%}')
print(f'  Reasoning:      {conservative.reasoning}')
print()
print('AGGRESSIVE AGENT:')
print(f'  Recommendation: {aggressive.recommendation}')
print(f'  Confidence:     {aggressive.confidence:.0%}')
print(f'  Reasoning:      {aggressive.reasoning}')

## 2. Detect disagreement

In [ ]:
def opinions_agree(a: AgentOpinion, b: AgentOpinion, threshold: float = 0.7) -> bool:
    """Check if two opinions are substantively similar using a quick LLM call."""
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{
            'role': 'user',
            'content': f'Do these two recommendations agree? Answer only yes or no.\nA: {a.recommendation}\nB: {b.recommendation}'
        }],
        max_tokens=5,
    )
    return 'yes' in resp.choices[0].message.content.lower()


agree = opinions_agree(conservative, aggressive)
print(f'Agents agree: {agree}')
if not agree:
    print('→ Escalating to human arbitration')

## 3. Human arbitration (simulated)

In [ ]:
# In production this would open a Gradio modal, send a Slack message, or email a manager
# Here we simulate the human decision
human_decision = {
    'winner': 'aggressive',
    'reasoning': 'The Q3 window is too valuable to miss. We can patch issues post-launch.',
    'directive': 'Proceed with Q3 launch. Conservative concerns about readiness should be addressed via a parallel launch checklist.'
}

print('HUMAN DECISION:')
print(f'  Winner:    {human_decision["winner"]} agent')
print(f'  Reasoning: {human_decision["reasoning"]}')
print(f'  Directive: {human_decision["directive"]}')

## 4. Both agents update and produce final synthesis

In [ ]:
synthesis_prompt = f"""You are synthesising a final recommendation from a multi-agent debate.

Question: {QUESTION}

Conservative agent said: {conservative.recommendation} ({conservative.reasoning})
Aggressive agent said: {aggressive.recommendation} ({aggressive.reasoning})

Human arbitrator decided: {human_decision['directive']}

Write a final action plan (3-5 bullet points) that incorporates the human's decision while
addressing the conservative agent's concerns as parallel tasks."""

resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': synthesis_prompt}],
    max_tokens=300,
)
print('FINAL SYNTHESIS:')
print(resp.choices[0].message.content)

## Exercise
Build a full arbitration system for 3 agents (not 2).  
When all 3 disagree, present all opinions to human with a summary of the key points of contention.  
After human decision, all 3 agents produce revised opinions that incorporate the directive.

**Bonus:** Track a running history of arbitration decisions and use it to pre-bias future agents.

In [ ]:
# Your code here
